# SQL for Data Analysis Bootcamp

A hands-on notebook covering **30 practical SQL concepts** for data analysis. Each topic has two cells: a Markdown explanation and a runnable code cell. SQL queries include `--` comments to explain their logic.

**Environment:** This notebook executes SQL using Python's built-in SQLite engine and displays results with pandas. It does not require installing a database server. Most SQL concepts transfer to PostgreSQL, MySQL, SQL Server, and other systems, but date functions and some syntax vary by dialect.

Run the cells from top to bottom. The first code cell creates and populates a sample relational dataset. Later lessons build toward a monthly business report.

## Learning path
1. Query and filter rows
2. Aggregate, group, and categorize data
3. Join and combine tables
4. Use subqueries, CTEs, and window functions
5. Validate data and build a business report

## 1. Database Setup and Tables

SQL (Structured Query Language) is used to retrieve, summarize, and manage data stored in relational databases. This bootcamp uses **SQLite**, which runs locally and needs no database server.

The sample schema has four tables: `customers`, `products`, `orders`, and `order_items`. Their keys connect customer records to orders and orders to purchased products. This first code cell creates and populates the database; later lessons query it.

**Dialect note:** Most examples use broadly familiar SQL. Date functions and a few details differ among SQLite, PostgreSQL, MySQL, SQL Server, and other databases.

In [ ]:
import sqlite3  # Import Python's built-in SQLite database driver.
import pandas as pd  # Import pandas so query results display as readable tables.

conn = sqlite3.connect(":memory:")  # Create a temporary database that exists only in memory.
conn.executescript("""
-- Create one row per customer and give each customer a unique ID.
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY, -- Unique customer identifier.
    customer_name TEXT NOT NULL, -- Customer's display name.
    city TEXT, -- Customer's city; it can be missing.
    segment TEXT NOT NULL, -- Customer segment, such as Consumer or Business.
    signup_date TEXT -- Signup date stored as ISO text (YYYY-MM-DD).
);

-- Store one row per product.
CREATE TABLE products (
    product_id INTEGER PRIMARY KEY, -- Unique product identifier.
    product_name TEXT NOT NULL, -- Product's name.
    category TEXT NOT NULL, -- Product category.
    unit_price REAL NOT NULL -- Current illustrative unit price.
);

-- Store the overall order header, one row per order.
CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY, -- Unique order identifier.
    customer_id INTEGER NOT NULL, -- Customer who placed the order.
    order_date TEXT NOT NULL, -- Order date stored as ISO text.
    status TEXT NOT NULL, -- Order status, such as Completed or Cancelled.
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id) -- Link each order to a customer.
);

-- Store line items because an order can contain multiple products.
CREATE TABLE order_items (
    item_id INTEGER PRIMARY KEY, -- Unique line-item identifier.
    order_id INTEGER NOT NULL, -- Order containing this line item.
    product_id INTEGER NOT NULL, -- Product purchased on this line.
    quantity INTEGER NOT NULL, -- Number of units on the line.
    unit_price REAL NOT NULL, -- Price charged per unit on this line.
    FOREIGN KEY (order_id) REFERENCES orders(order_id), -- Link item to its order.
    FOREIGN KEY (product_id) REFERENCES products(product_id) -- Link item to its product.
);
""")  # Execute all table definitions at once.

conn.executemany("INSERT INTO customers VALUES (?, ?, ?, ?, ?)", [  # Insert sample customer rows safely with parameters.
    (1, "Ava Chen", "London", "Consumer", "2024-11-12"),  # Add customer 1.
    (2, "Noah Smith", "Manchester", "Business", "2024-12-03"),  # Add customer 2.
    (3, "Mia Patel", "London", "Consumer", "2025-01-15"),  # Add customer 3.
    (4, "Leo Martin", "Bristol", "Small Business", "2025-02-02"),  # Add customer 4.
    (5, "Ella Brown", None, "Consumer", "2025-02-18"),  # Add a customer with an unknown city.
    (6, "Amir Khan", "Leeds", "Business", "2025-03-06"),  # Add customer 6.
    (7, "Zoe Wilson", "Bristol", "Consumer", "2025-03-22"),  # Add customer 7.
    (8, "Luca Rossi", "London", "Small Business", "2025-04-01"),  # Add customer 8.
])  # Finish loading customers.

conn.executemany("INSERT INTO products VALUES (?, ?, ?, ?)", [  # Insert product catalog rows.
    (101, "Laptop", "Computers", 900.0),  # Add the laptop product.
    (102, "Phone", "Electronics", 500.0),  # Add the phone product.
    (103, "Tablet", "Electronics", 300.0),  # Add the tablet product.
    (104, "Monitor", "Accessories", 220.0),  # Add the monitor product.
    (105, "Keyboard", "Accessories", 45.0),  # Add the keyboard product.
    (106, "Desk", "Furniture", 180.0),  # Add the desk product.
])  # Finish loading products.

conn.executemany("INSERT INTO orders VALUES (?, ?, ?, ?)", [  # Insert order header rows.
    (1001, 1, "2025-01-05", "Completed"),  # Add order 1001.
    (1002, 2, "2025-01-12", "Completed"),  # Add order 1002.
    (1003, 1, "2025-01-20", "Cancelled"),  # Add a cancelled order.
    (1004, 3, "2025-02-03", "Completed"),  # Add order 1004.
    (1005, 4, "2025-02-15", "Completed"),  # Add order 1005.
    (1006, 5, "2025-02-20", "Pending"),  # Add a pending order.
    (1007, 6, "2025-03-02", "Completed"),  # Add order 1007.
    (1008, 2, "2025-03-09", "Completed"),  # Add order 1008.
    (1009, 7, "2025-03-17", "Completed"),  # Add order 1009.
    (1010, 8, "2025-04-04", "Completed"),  # Add order 1010.
    (1011, 3, "2025-04-11", "Completed"),  # Add order 1011.
    (1012, 4, "2025-04-21", "Cancelled"),  # Add a cancelled order.
    (1013, 1, "2025-05-06", "Completed"),  # Add order 1013.
    (1014, 6, "2025-05-13", "Completed"),  # Add order 1014.
    (1015, 5, "2025-05-27", "Completed"),  # Add order 1015.
    (1016, 7, "2025-06-08", "Pending"),  # Add a pending order.
    (1017, 8, "2025-06-15", "Completed"),  # Add order 1017.
    (1018, 2, "2025-06-24", "Completed"),  # Add order 1018.
])  # Finish loading orders.

conn.executemany("INSERT INTO order_items VALUES (?, ?, ?, ?, ?)", [  # Insert order line items.
    (1, 1001, 101, 1, 880.0), (2, 1001, 105, 2, 40.0),  # Order 1001 contains a laptop and keyboards.
    (3, 1002, 102, 2, 480.0), (4, 1002, 104, 1, 210.0),  # Order 1002 contains phones and a monitor.
    (5, 1003, 106, 1, 180.0),  # Cancelled order 1003 contains a desk.
    (6, 1004, 103, 2, 290.0), (7, 1004, 105, 1, 45.0),  # Order 1004 contains tablets and a keyboard.
    (8, 1005, 101, 1, 900.0), (9, 1005, 104, 2, 220.0),  # Order 1005 contains a laptop and monitors.
    (10, 1006, 102, 1, 500.0),  # Pending order 1006 contains a phone.
    (11, 1007, 101, 2, 870.0), (12, 1007, 105, 3, 42.0),  # Order 1007 contains laptops and keyboards.
    (13, 1008, 103, 1, 300.0), (14, 1008, 106, 2, 175.0),  # Order 1008 contains a tablet and desks.
    (15, 1009, 102, 3, 490.0),  # Order 1009 contains phones.
    (16, 1010, 104, 2, 215.0), (17, 1010, 105, 2, 44.0),  # Order 1010 contains monitors and keyboards.
    (18, 1011, 101, 1, 910.0), (19, 1011, 102, 1, 495.0),  # Order 1011 contains a laptop and phone.
    (20, 1012, 106, 2, 180.0),  # Cancelled order 1012 contains desks.
    (21, 1013, 101, 1, 920.0), (22, 1013, 103, 2, 295.0),  # Order 1013 contains a laptop and tablets.
    (23, 1014, 102, 2, 505.0), (24, 1014, 104, 1, 220.0),  # Order 1014 contains phones and a monitor.
    (25, 1015, 105, 4, 45.0), (26, 1015, 106, 1, 180.0),  # Order 1015 contains keyboards and a desk.
    (27, 1016, 103, 3, 305.0),  # Pending order 1016 contains tablets.
    (28, 1017, 101, 1, 895.0), (29, 1017, 104, 1, 215.0),  # Order 1017 contains a laptop and monitor.
    (30, 1018, 102, 2, 495.0), (31, 1018, 105, 1, 45.0),  # Order 1018 contains phones and a keyboard.
])  # Finish loading line items.
conn.commit()  # Save the inserted sample records.

def run_query(sql):  # Define a helper that runs a SQL SELECT query and displays its output.
    result = pd.read_sql_query(sql, conn)  # Execute SQL and load results into a pandas DataFrame.
    print(result.to_string(index=False))  # Print the result without the pandas row index.
    return result  # Return the DataFrame for additional analysis if needed.

# Run a first query to inspect the sample order table.
run_query("""
-- Select the first five orders in the database.
SELECT order_id, customer_id, order_date, status
FROM orders
ORDER BY order_id
LIMIT 5;
""")  # End the first query.

## 2. SELECT: Choose Columns

`SELECT` chooses which columns to return, and `FROM` identifies the table. This is the starting point for most analytical queries.

Avoid `SELECT *` in production queries when you know which fields you need: explicit column lists make queries clearer and reduce unnecessary data transfer.

In [ ]:
run_query("""  -- Select only the fields needed for a compact order overview.
SELECT order_id, order_date, status -- Return the order ID, date, and status.
FROM orders; -- Read rows from the orders table.
""")  # Execute the SELECT query and display its result.

## 3. Aliases and Calculated Columns

An alias renames a column in the query result with `AS`. SQL can also calculate new columns from existing values, such as `quantity * unit_price`.

Aliases make outputs easier to understand and are especially helpful for reports, charts, and exported analysis tables.

In [ ]:
run_query("""  -- Calculate extended line revenue and give it a readable name.
SELECT
    item_id, -- Keep the line-item identifier.
    order_id, -- Identify the order containing the line.
    quantity, -- Return the number of units.
    unit_price, -- Return the price charged per unit.
    quantity * unit_price AS line_revenue -- Multiply quantity by price and name the calculated result.
FROM order_items -- Read line-item records.
ORDER BY order_id, item_id -- Show order lines in a predictable sequence.
LIMIT 10; -- Keep the preview small.
""")  # Execute the calculated-column query.

## 4. WHERE: Filter Rows

`WHERE` filters individual rows before grouping or aggregation. Comparison operators include `=`, `<>`, `>`, `>=`, `<`, and `<=`.

For example, analysts commonly filter by order status, reporting date, product category, or minimum transaction amount.

In [ ]:
run_query("""  -- Return only completed orders after February 1, 2025.
SELECT order_id, customer_id, order_date, status -- Choose columns to show.
FROM orders -- Read the order records.
WHERE status = 'Completed' -- Keep only completed orders.
  AND order_date >= '2025-02-01' -- Keep orders on or after the chosen date.
ORDER BY order_date; -- Display them chronologically.
""")  # Execute the filter query.

## 5. AND, OR, and NOT

Use `AND` when every condition must be true, `OR` when at least one condition must be true, and `NOT` to negate a condition.

Parentheses make mixed conditions easier to understand and prevent logical mistakes. For example, a rule like “completed orders from London or Bristol” needs the location conditions grouped together.

In [ ]:
run_query("""  -- Find completed orders belonging to customers in London or Bristol.
SELECT o.order_id, c.customer_name, c.city, o.status -- Return order and customer details.
FROM orders AS o -- Give the orders table the short alias o.
JOIN customers AS c ON o.customer_id = c.customer_id -- Match each order with its customer.
WHERE o.status = 'Completed' -- Require the order to be completed.
  AND (c.city = 'London' OR c.city = 'Bristol') -- Accept either of the two cities.
ORDER BY c.city, o.order_date; -- Sort by city and then date.
""")  # Execute the query with combined conditions.

## 6. IN and BETWEEN

`IN` checks whether a value matches any item in a list. `BETWEEN` checks whether a value lies within an inclusive range.

These operators make category filters and numeric or date ranges easier to read. Remember that `BETWEEN` includes both endpoints.

In [ ]:
run_query("""  -- Find orders in a chosen status set and a date range.
SELECT order_id, order_date, status -- Return useful order fields.
FROM orders -- Read the order table.
WHERE status IN ('Completed', 'Pending') -- Accept either listed status.
  AND order_date BETWEEN '2025-03-01' AND '2025-04-30' -- Include both date boundaries.
ORDER BY order_date; -- Sort the qualifying orders chronologically.
""")  # Execute the IN and BETWEEN query.

## 7. LIKE: Pattern Matching

`LIKE` matches text patterns. In SQL, `%` means any sequence of characters and `_` means one character.

Pattern matching is useful for searching names or categories. Case sensitivity and wildcard behavior can vary by database and collation; SQLite's default ASCII `LIKE` behavior is often case-insensitive.

In [ ]:
run_query("""  -- Search for product names containing the text 'phone'.
SELECT product_id, product_name, category -- Return product details.
FROM products -- Read the product catalog.
WHERE product_name LIKE '%phone%' -- Match names with any text before or after 'phone'.
ORDER BY product_name; -- Sort matching product names.
""")  # Execute the pattern search.

## 8. ORDER BY and LIMIT

`ORDER BY` sorts a result set. `ASC` is ascending and `DESC` is descending. `LIMIT` restricts how many rows are returned in SQLite and PostgreSQL (other databases may use different syntax).

Always add a meaningful sort key when selecting top or bottom records; otherwise, which rows appear first is not guaranteed.

In [ ]:
run_query("""  -- Show the five highest-value line items.
SELECT item_id, order_id, quantity * unit_price AS line_revenue -- Calculate each line's extended revenue.
FROM order_items -- Read order lines.
ORDER BY line_revenue DESC, item_id ASC -- Rank by revenue and use item ID to break ties.
LIMIT 5; -- Return only the first five rows.
""")  # Execute the top-five query.

## 9. DISTINCT: Unique Values

`DISTINCT` removes duplicate rows from the selected columns. It is useful for exploring categories, finding unique customers, or checking which statuses appear in the data.

When multiple columns are selected, `DISTINCT` applies to the combination of their values rather than each column independently.

In [ ]:
run_query("""  -- List the unique order statuses and customer segments found in the database.
SELECT DISTINCT status -- Return each status once.
FROM orders -- Read order status values.
ORDER BY status; -- Sort the unique statuses.
""")  # Execute the distinct-status query.

run_query("""  -- List unique customer city and segment combinations.
SELECT DISTINCT city, segment -- Keep each distinct pair of city and customer segment.
FROM customers -- Read the customer table.
ORDER BY city, segment; -- Sort the combinations for readability.
""")  # Execute the multi-column DISTINCT query.

## 10. NULL: Missing Values

SQL uses `NULL` to represent missing or unknown values. Check them with `IS NULL` or `IS NOT NULL`; comparisons such as `city = NULL` do not work as intended.

`COALESCE(value, fallback)` returns the first non-NULL expression. Use a fallback only when it makes sense for the business definition.

In [ ]:
run_query("""  -- Find customers whose city is missing and display a safe label.
SELECT
    customer_id, -- Return the customer's ID.
    customer_name, -- Return the customer's name.
    COALESCE(city, 'Unknown') AS city_label -- Replace a missing city only in the query output.
FROM customers -- Read customer records.
WHERE city IS NULL; -- Keep only customers with missing city values.
""")  # Execute the missing-value query.

## 11. Aggregate Functions

Aggregate functions summarize many rows into a single result: `COUNT`, `SUM`, `AVG`, `MIN`, and `MAX`.

`COUNT(*)` counts rows, while `COUNT(column)` counts non-NULL values in that column. `SUM` and `AVG` ignore NULL values. Be clear about the unit being counted—orders, order lines, or customers.

In [ ]:
run_query("""  -- Summarize the orders table.
SELECT
    COUNT(*) AS order_count, -- Count all order rows.
    COUNT(DISTINCT customer_id) AS distinct_customers, -- Count customers who placed at least one order.
    MIN(order_date) AS first_order_date, -- Find the earliest order date.
    MAX(order_date) AS last_order_date -- Find the latest order date.
FROM orders; -- Calculate the metrics over the full orders table.
""")  # Execute overall order metrics.

run_query("""  -- Summarize the quantities and prices in order lines.
SELECT
    SUM(quantity) AS units_sold, -- Add all unit quantities.
    ROUND(AVG(unit_price), 2) AS average_unit_price, -- Calculate the mean line-item unit price.
    MIN(unit_price) AS lowest_unit_price, -- Find the lowest price charged.
    MAX(unit_price) AS highest_unit_price -- Find the highest price charged.
FROM order_items; -- Aggregate across all line items.
""")  # Execute the line-item metrics query.

## 12. GROUP BY

`GROUP BY` calculates aggregates separately for each category or combination of categories. It is similar to pandas' `groupby()`.

Every selected column must either be grouped, aggregated, or accepted by the database's rules. Keep the grouping level in mind: grouping by region gives one row per region, while grouping by region and month gives one row per region-month pair.

In [ ]:
run_query("""  -- Count orders and distinct customers for each status.
SELECT
    status, -- Define the grouping category.
    COUNT(*) AS order_count, -- Count order rows within each status.
    COUNT(DISTINCT customer_id) AS customer_count -- Count distinct customers within each status.
FROM orders -- Read the order table.
GROUP BY status -- Produce one summary row per status.
ORDER BY order_count DESC; -- Rank statuses by order count.
""")  # Execute the grouped aggregation.

## 13. HAVING: Filter Groups

`WHERE` filters rows before aggregation, while `HAVING` filters groups after aggregation. Use `HAVING` for conditions such as “only products with more than 10 units sold” or “regions whose revenue exceeds a target.”

A common query sequence is `FROM` → `WHERE` → `GROUP BY` → `HAVING` → `ORDER BY` → `LIMIT`.

In [ ]:
run_query("""  -- Find products with at least three units sold across completed orders.
SELECT
    p.product_name, -- Display the product name.
    SUM(oi.quantity) AS units_sold -- Sum the quantity sold per product.
FROM order_items AS oi -- Start with individual order lines.
JOIN orders AS o ON oi.order_id = o.order_id -- Attach each line to its order status.
JOIN products AS p ON oi.product_id = p.product_id -- Attach each line to its product.
WHERE o.status = 'Completed' -- Exclude pending and cancelled orders before summing.
GROUP BY p.product_name -- Summarize one row per product.
HAVING SUM(oi.quantity) >= 3 -- Keep only products with at least three sold units.
ORDER BY units_sold DESC; -- Rank qualifying products by units sold.
""")  # Execute the HAVING query.

## 14. CASE: Conditional Logic

`CASE` implements if/then/else logic in SQL. It can create categories, flags, or conditional metrics in a query.

Conditions are checked in order, and the first matching `WHEN` branch is returned. Put the most specific thresholds first when creating numeric bands.

In [ ]:
run_query("""  -- Categorize each product by its current catalog price.
SELECT
    product_name, -- Return the product label.
    unit_price, -- Return its listed price.
    CASE
        WHEN unit_price >= 700 THEN 'Premium' -- Assign Premium to prices of 700 or more.
        WHEN unit_price >= 200 THEN 'Mid-range' -- Assign Mid-range to prices from 200 up to 700.
        ELSE 'Budget' -- Label all lower prices as Budget.
    END AS price_band -- Name the calculated category.
FROM products -- Read the product catalog.
ORDER BY unit_price DESC; -- Show expensive products first.
""")  # Execute the CASE expression query.

## 15. String Functions

SQL string functions are useful for cleaning and formatting text. Common examples include `LOWER`, `UPPER`, `TRIM`, and concatenation.

Function names and concatenation syntax vary by database. This example uses SQLite's `LOWER`, `UPPER`, `TRIM`, and `||` operator.

In [ ]:
run_query("""  -- Show a few examples of text normalization and display formatting.
SELECT
    customer_name, -- Return the original customer name.
    UPPER(customer_name) AS name_uppercase, -- Convert the name to uppercase.
    LOWER(segment) AS segment_lowercase, -- Convert the segment to lowercase.
    COALESCE(TRIM(city), 'Unknown') AS cleaned_city, -- Remove extra edge spaces or display Unknown for NULL.
    customer_name || ' (' || segment || ')' AS customer_label -- Combine name and segment into a label.
FROM customers -- Read customer records.
ORDER BY customer_name; -- Sort names alphabetically.
""")  # Execute the text-function examples.

## 16. Date Functions

Date analysis is common in reporting, cohort analysis, and trend analysis. SQLite stores these sample dates as ISO-formatted text and provides functions such as `date()`, `strftime()`, and `julianday()`.

Other databases may use different functions, such as `DATE_TRUNC` in PostgreSQL. Confirm the dialect before moving a query between systems.

In [ ]:
run_query("""  -- Extract year and month fields using SQLite date functions.
SELECT
    order_id, -- Return the order identifier.
    order_date, -- Show the original ISO date.
    strftime('%Y', order_date) AS order_year, -- Extract the four-digit year.
    strftime('%m', order_date) AS order_month, -- Extract the two-digit month.
    date(order_date, '+30 days') AS date_after_30_days -- Calculate a date 30 days later.
FROM orders -- Read order records.
ORDER BY order_date -- Sort by order date.
LIMIT 8; -- Keep the output concise.
""")  # Execute the SQLite date-function query.

## 17. INNER JOIN

`INNER JOIN` returns rows with matching keys on both sides. Joins let analysts combine orders with customer attributes, or order items with product names.

Always join tables using the appropriate key. If one order has several line items, joining orders to order items will create several rows per order; this is expected and affects how you should count or aggregate.

In [ ]:
run_query("""  -- Attach customer names to each order that has a matching customer.
SELECT
    o.order_id, -- Return the order identifier.
    o.order_date, -- Return the order date.
    c.customer_name, -- Add the customer name from the customer table.
    c.city, -- Add the customer's city.
    o.status -- Return the order status.
FROM orders AS o -- Use o as a short alias for orders.
INNER JOIN customers AS c ON o.customer_id = c.customer_id -- Keep rows with matching customer IDs.
ORDER BY o.order_date -- Sort by date.
LIMIT 10; -- Return a manageable preview.
""")  # Execute the INNER JOIN query.

## 18. LEFT JOIN and Unmatched Records

`LEFT JOIN` keeps every row from the left table, even if there is no matching row on the right. Columns from the unmatched right side become NULL.

This is useful for finding customers who have never ordered or products that have never appeared in an order line. Be careful when adding filters on the right table in `WHERE`, because those filters can accidentally remove unmatched rows.

In [ ]:
run_query("""  -- Find customers with no matching order record.
SELECT
    c.customer_id, -- Return the customer's ID.
    c.customer_name, -- Return the customer's name.
    c.city -- Return the customer's city.
FROM customers AS c -- Keep every customer as the left-side row.
LEFT JOIN orders AS o ON c.customer_id = o.customer_id -- Match any orders belonging to the customer.
WHERE o.order_id IS NULL -- Keep only customers for whom no order matched.
ORDER BY c.customer_name; -- Sort customer names.
""")  # Execute the anti-match pattern using LEFT JOIN.

## 19. Join Multiple Tables and Calculate Revenue

To calculate sales revenue accurately, analysts often join order lines to orders and products. Here, line revenue is the quantity multiplied by the price actually recorded on the line.

Filter order status before adding revenue if the business definition excludes cancelled or pending orders. Also check whether the analysis needs gross sales, net sales after discounts, or another measure.

In [ ]:
run_query("""  -- Calculate completed-order revenue by product category.
SELECT
    p.category, -- Group the result by product category.
    SUM(oi.quantity) AS units_sold, -- Sum units across qualifying line items.
    ROUND(SUM(oi.quantity * oi.unit_price), 2) AS revenue -- Calculate and sum line-item revenue.
FROM order_items AS oi -- Start from order lines.
JOIN orders AS o ON oi.order_id = o.order_id -- Attach the status of each order.
JOIN products AS p ON oi.product_id = p.product_id -- Attach product category.
WHERE o.status = 'Completed' -- Include completed orders only.
GROUP BY p.category -- Summarize rows by product category.
ORDER BY revenue DESC; -- Rank categories by revenue.
""")  # Execute the three-table revenue query.

## 20. Subqueries

A subquery is a query nested inside another query. It can calculate a threshold, identify a set of keys, or provide a derived table for the outer query.

Use subqueries when they make the logic easier to understand. A correlated subquery refers to values from the outer query and may be executed repeatedly by some query plans, so check performance for large data.

In [ ]:
run_query("""  -- Find completed orders whose total line revenue is above the average completed-order revenue.
SELECT
    order_totals.order_id, -- Return the qualifying order ID.
    order_totals.order_revenue -- Return its calculated revenue.
FROM (
    SELECT
        o.order_id, -- Identify each order.
        SUM(oi.quantity * oi.unit_price) AS order_revenue -- Sum revenue across each order's line items.
    FROM orders AS o -- Start with order headers.
    JOIN order_items AS oi ON o.order_id = oi.order_id -- Attach the order's line items.
    WHERE o.status = 'Completed' -- Keep completed orders for this calculation.
    GROUP BY o.order_id -- Calculate one total per order.
) AS order_totals -- Name the derived table so the outer query can reference it.
WHERE order_totals.order_revenue > (
    SELECT AVG(order_revenue) -- Calculate the average order total.
    FROM (
        SELECT SUM(oi2.quantity * oi2.unit_price) AS order_revenue -- Calculate one revenue total per order.
        FROM orders AS o2 -- Start from order headers.
        JOIN order_items AS oi2 ON o2.order_id = oi2.order_id -- Attach each order's items.
        WHERE o2.status = 'Completed' -- Use the same completed-order population.
        GROUP BY o2.order_id -- Produce one total per order for the average.
    ) AS average_source -- Name the inner derived table.
)
ORDER BY order_totals.order_revenue DESC; -- Rank qualifying orders by revenue.
""")  # Execute the nested subquery example.

## 21. Common Table Expressions (CTEs)

A Common Table Expression, or **CTE**, is a named query defined with `WITH`. It lets you break a complicated query into readable steps.

CTEs are particularly helpful for analytics workflows: calculate order totals first, then summarize or rank those totals in a later step.

In [ ]:
run_query("""  -- Calculate order totals first, then rank them in the outer query.
WITH completed_order_totals AS ( -- Define a reusable named result for completed-order totals.
    SELECT
        o.order_id, -- Keep the order identifier.
        o.customer_id, -- Keep the customer identifier for later grouping.
        SUM(oi.quantity * oi.unit_price) AS order_revenue -- Calculate each completed order's revenue.
    FROM orders AS o -- Start from order headers.
    JOIN order_items AS oi ON o.order_id = oi.order_id -- Join order lines.
    WHERE o.status = 'Completed' -- Exclude non-completed orders.
    GROUP BY o.order_id, o.customer_id -- Produce one row per completed order.
)
SELECT
    order_id, -- Return the order ID.
    customer_id, -- Return the customer ID.
    order_revenue, -- Return calculated revenue.
    RANK() OVER (ORDER BY order_revenue DESC) AS revenue_rank -- Rank orders from highest to lowest revenue.
FROM completed_order_totals -- Query the CTE as if it were a table.
ORDER BY revenue_rank, order_id; -- Display the largest orders first.
""")  # Execute the CTE query.

## 22. UNION and UNION ALL

`UNION` combines compatible result sets and removes duplicate rows. `UNION ALL` combines them while keeping duplicates; it is usually faster when deduplication is unnecessary.

Each SELECT must return the same number of columns with compatible data types. The output column names normally come from the first SELECT.

In [ ]:
run_query("""  -- Combine London and Bristol customer lists while removing duplicate output rows.
SELECT city AS selected_city, customer_name -- Return city and customer name for the first group.
FROM customers -- Read the customer table.
WHERE city = 'London' -- Filter to London.
UNION -- Combine with the second query and remove duplicate result rows.
SELECT city AS selected_city, customer_name -- Return the same two columns for the second group.
FROM customers -- Read the customer table again.
WHERE city = 'Bristol' -- Filter to Bristol.
ORDER BY selected_city, customer_name; -- Sort the combined output.
""")  # Execute the UNION example.

run_query("""  -- Use UNION ALL when preserving every row from both queries is desired.
SELECT 'Completed' AS order_group, order_id -- Label rows from the completed-order query.
FROM orders -- Read the order table.
WHERE status = 'Completed' -- Select completed orders.
UNION ALL -- Append rows without deduplication.
SELECT 'Pending' AS order_group, order_id -- Label rows from the pending-order query.
FROM orders -- Read the order table again.
WHERE status = 'Pending' -- Select pending orders.
ORDER BY order_id; -- Sort the combined records by ID.
""")  # Execute the UNION ALL example.

## 23. Conditional Aggregation

Conditional aggregation calculates multiple metrics from the same group using `SUM(CASE WHEN ...)`. It is useful for a single summary row or grouped report that compares statuses or segments side by side.

This pattern is portable across many SQL databases and often avoids running separate queries for each category.

In [ ]:
run_query("""  -- Compare order counts by status in one result row.
SELECT
    COUNT(*) AS all_orders, -- Count every order.
    SUM(CASE WHEN status = 'Completed' THEN 1 ELSE 0 END) AS completed_orders, -- Count completed orders.
    SUM(CASE WHEN status = 'Pending' THEN 1 ELSE 0 END) AS pending_orders, -- Count pending orders.
    SUM(CASE WHEN status = 'Cancelled' THEN 1 ELSE 0 END) AS cancelled_orders -- Count cancelled orders.
FROM orders; -- Aggregate all order rows.
""")  # Execute the conditional aggregation query.

run_query("""  -- Calculate completed and non-completed line revenue by product category.
SELECT
    p.category, -- Group by category.
    SUM(CASE WHEN o.status = 'Completed' THEN oi.quantity * oi.unit_price ELSE 0 END) AS completed_revenue, -- Sum only completed-order revenue.
    SUM(CASE WHEN o.status <> 'Completed' THEN oi.quantity * oi.unit_price ELSE 0 END) AS other_status_revenue -- Sum revenue recorded on other statuses.
FROM order_items AS oi -- Begin with line items.
JOIN orders AS o ON oi.order_id = o.order_id -- Attach order status.
JOIN products AS p ON oi.product_id = p.product_id -- Attach category.
GROUP BY p.category -- Produce one row per category.
ORDER BY completed_revenue DESC; -- Rank by completed revenue.
""")  # Execute the grouped conditional aggregation.

## 24. Window Functions: ROW_NUMBER and RANK

Window functions calculate values across related rows while keeping each original row. Unlike `GROUP BY`, they do not collapse the rows into one per group.

`ROW_NUMBER()` assigns a unique sequence; `RANK()` gives ties the same rank and leaves gaps; `DENSE_RANK()` gives ties the same rank without gaps. Use `PARTITION BY` to restart ranking within each group.

In [ ]:
run_query("""  -- Rank products within each category by their catalog price.
SELECT
    product_name, -- Return the product name.
    category, -- Return its category.
    unit_price, -- Return its catalog price.
    ROW_NUMBER() OVER (PARTITION BY category ORDER BY unit_price DESC) AS row_number_in_category, -- Number products inside each category.
    RANK() OVER (PARTITION BY category ORDER BY unit_price DESC) AS price_rank, -- Rank equal prices with gaps after ties.
    DENSE_RANK() OVER (PARTITION BY category ORDER BY unit_price DESC) AS dense_price_rank -- Rank equal prices without gaps.
FROM products -- Read the product catalog.
ORDER BY category, unit_price DESC; -- Display each category in descending price order.
""")  # Execute the window ranking query.

## 25. Window Functions: LAG and LEAD

`LAG()` reads a value from a previous row, while `LEAD()` reads a value from a following row in the specified window order. They are useful for comparing a period with the prior period or inspecting successive events.

The order in `OVER (ORDER BY ...)` is essential. Without a deterministic order, the meaning of “previous” or “next” is unclear.

In [ ]:
run_query("""  -- Compare each customer's orders with their previous order chronologically.
SELECT
    customer_id, -- Identify the customer.
    order_id, -- Identify the current order.
    order_date, -- Show the current order date.
    LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date, order_id) AS previous_order_date, -- Read the previous order date for this customer.
    LEAD(order_date) OVER (PARTITION BY customer_id ORDER BY order_date, order_id) AS next_order_date -- Read the next order date for this customer.
FROM orders -- Read all order records.
ORDER BY customer_id, order_date, order_id; -- Display each customer's orders in time order.
""")  # Execute the LAG/LEAD query.

## 26. Running Totals and Moving Averages

A running total accumulates a value across an ordered set of rows. A moving average summarizes a defined window of nearby rows.

Window frames determine which rows contribute. For a running total, an explicit frame from unbounded preceding through the current row communicates the intended behavior and avoids ambiguity around peer rows.

In [ ]:
run_query("""  -- Build a running revenue total across completed orders by date.
WITH daily_revenue AS ( -- Define the daily aggregation before calculating the running total.
    SELECT
        o.order_date, -- Keep the calendar date.
        SUM(oi.quantity * oi.unit_price) AS revenue -- Add line revenue for completed orders.
    FROM orders AS o -- Start from order headers.
    JOIN order_items AS oi ON o.order_id = oi.order_id -- Attach line items.
    WHERE o.status = 'Completed' -- Keep only completed orders.
    GROUP BY o.order_date -- Produce one row per date.
)
SELECT
    order_date, -- Return the date.
    revenue, -- Return the day's revenue.
    SUM(revenue) OVER (ORDER BY order_date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_revenue, -- Add each day to all prior days.
    AVG(revenue) OVER (ORDER BY order_date ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS moving_average_3_rows -- Average current and previous two rows where available.
FROM daily_revenue -- Query the daily totals.
ORDER BY order_date; -- Ensure results are chronological.
""")  # Execute the running-total query.

## 27. Percent of Total with Window Functions

Window aggregates can compare each row or group with the overall total without collapsing the rows. A common metric is each product's percentage contribution to revenue.

`SUM(metric) OVER ()` calculates a total across the whole result. Use `NULLIF(denominator, 0)` when a division-by-zero case is possible, and remember that filters applied before the window change the total's population.

In [ ]:
run_query("""  -- Calculate each product's share of completed-order revenue.
WITH product_revenue AS ( -- Aggregate the completed-order sales to one row per product.
    SELECT
        p.product_name, -- Keep the product name.
        SUM(oi.quantity * oi.unit_price) AS revenue -- Sum revenue from this product's order lines.
    FROM order_items AS oi -- Start from order lines.
    JOIN orders AS o ON oi.order_id = o.order_id -- Join to order status.
    JOIN products AS p ON oi.product_id = p.product_id -- Join to product name.
    WHERE o.status = 'Completed' -- Exclude pending and cancelled orders.
    GROUP BY p.product_name -- Produce one row per product.
)
SELECT
    product_name, -- Return product name.
    revenue, -- Return the product's revenue total.
    ROUND(100.0 * revenue / NULLIF(SUM(revenue) OVER (), 0), 2) AS revenue_share_percent -- Calculate contribution to all listed product revenue.
FROM product_revenue -- Use the product-level totals.
ORDER BY revenue DESC; -- Rank products by revenue.
""")  # Execute the contribution-to-total query.

## 28. EXISTS and NOT EXISTS

`EXISTS` checks whether a subquery returns at least one row; `NOT EXISTS` checks that it returns none. These are useful for questions such as “Which customers have at least one completed order?” or “Which customers have no orders?”

Compared with joining, `EXISTS` can express an existence condition without producing repeated customer rows when a customer has multiple qualifying orders.

In [ ]:
run_query("""  -- Return customers who have at least one completed order.
SELECT
    c.customer_id, -- Return the customer ID.
    c.customer_name -- Return the customer name.
FROM customers AS c -- Begin with every customer.
WHERE EXISTS ( -- Keep the customer only if a matching row exists.
    SELECT 1 -- The value is irrelevant; only existence matters.
    FROM orders AS o -- Search order rows.
    WHERE o.customer_id = c.customer_id -- Match orders to the current customer.
      AND o.status = 'Completed' -- Require a completed order.
)
ORDER BY c.customer_name; -- Sort the qualifying customers.
""")  # Execute the EXISTS query.

run_query("""  -- Return customers who have no order of any status.
SELECT
    c.customer_id, -- Return the customer ID.
    c.customer_name -- Return the customer name.
FROM customers AS c -- Begin with every customer.
WHERE NOT EXISTS ( -- Keep the customer only if no matching row exists.
    SELECT 1 -- Only row existence matters.
    FROM orders AS o -- Search the order table.
    WHERE o.customer_id = c.customer_id -- Match orders to the current customer.
)
ORDER BY c.customer_name; -- Sort the customers without orders.
""")  # Execute the NOT EXISTS query.

## 29. Data Quality Checks

SQL is useful for validating data before analysis. Typical checks include finding duplicate identifiers, missing values, unexpected categories, negative quantities, or orphaned foreign keys.

This section demonstrates reusable diagnostic queries. A result with zero rows can be a good sign, but the checks should match the actual business rules and schema.

In [ ]:
run_query("""  -- Find duplicate order IDs, if any exist.
SELECT order_id, COUNT(*) AS row_count -- Return each repeated ID and its frequency.
FROM orders -- Check the order table.
GROUP BY order_id -- Group records by identifier.
HAVING COUNT(*) > 1; -- Keep IDs that occur more than once.
""")  # Execute the duplicate-key check.

run_query("""  -- Look for invalid non-positive quantities in the order lines.
SELECT item_id, order_id, product_id, quantity -- Show the line details for investigation.
FROM order_items -- Check the order-item table.
WHERE quantity <= 0; -- Keep rows with quantities that violate the positive-quantity rule.
""")  # Execute the quantity validation check.

run_query("""  -- Check whether any order line references a missing product row.
SELECT oi.item_id, oi.product_id -- Return the line item and unmatched product ID.
FROM order_items AS oi -- Begin with the line items.
LEFT JOIN products AS p ON oi.product_id = p.product_id -- Try to match each line to the product catalog.
WHERE p.product_id IS NULL; -- Keep only orphaned product references.
""")  # Execute the referential-integrity check.

## 30. Mini Project: Monthly Business Report

This final query combines joins, filtering, grouping, a CTE, and a window function to produce a monthly completed-sales report.

The report includes monthly revenue, unique orders, unique customers, and each month's share of the total revenue in the period. Always ensure the chosen date range, order statuses, and definition of revenue match your business requirements.

In [ ]:
run_query("""  -- Build a monthly completed-sales report with each month's revenue share.
WITH monthly_metrics AS ( -- Define the monthly summary as a named intermediate result.
    SELECT
        strftime('%Y-%m', o.order_date) AS sales_month, -- Convert the ISO order date to a year-month label.
        SUM(oi.quantity * oi.unit_price) AS revenue, -- Add line revenue for completed orders.
        COUNT(DISTINCT o.order_id) AS completed_orders, -- Count distinct completed orders.
        COUNT(DISTINCT o.customer_id) AS active_customers -- Count customers who ordered that month.
    FROM orders AS o -- Start with order headers.
    JOIN order_items AS oi ON o.order_id = oi.order_id -- Attach each order's line items.
    WHERE o.status = 'Completed' -- Include completed orders only.
      AND o.order_date >= '2025-01-01' -- Set the inclusive lower date limit.
      AND o.order_date < '2025-07-01' -- Set the exclusive upper date limit.
    GROUP BY strftime('%Y-%m', o.order_date) -- Aggregate one row per calendar month.
)
SELECT
    sales_month, -- Return the month label.
    ROUND(revenue, 2) AS revenue, -- Return monthly revenue rounded to two decimal places.
    completed_orders, -- Return completed order count.
    active_customers, -- Return unique customer count for the month.
    ROUND(100.0 * revenue / NULLIF(SUM(revenue) OVER (), 0), 2) AS revenue_share_percent -- Calculate share of revenue over the report period.
FROM monthly_metrics -- Query the monthly summary.
ORDER BY sales_month; -- Display months chronologically.
""")  # Execute the final monthly report query.